<a href="https://colab.research.google.com/github/Pillapavan/Ai-Engineering/blob/main/Day1_Semantic_Search_RAG_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers scikit-learn

In [ ]:
from sentence_transformers import SentenceTransformer #Text → Embedding vector
from sklearn.metrics.pairwise import cosine_similarity # Vector A + Vector B -> Similarity score
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2") # all-MiniLM-L6-v2 -> 384-dimensional embedding

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
documents = '''
    "Employees are entitled to 20 days of annual leave each year.",

    "Employees can reset their company password through the security portal. "
    "Password changes require identity verification.",

    "Employees can work remotely up to three days per week, subject to manager approval.",

    "Employees can contact the HR department for questions about employee benefits.",

    "The company provides health insurance and retirement benefits to eligible employees."
'''

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=5,
    length_function=len  # Uses Python's standard character counting
)

documents = text_splitter.split_text(documents)

for i, chunk in enumerate(documents):
    print(f"\n--- Chunk {i + 1} ---")
    print(chunk)


--- Chunk 1 ---
"Employees are entitled to 20 days of annual leave each year.",

--- Chunk 2 ---
"Employees can reset their company password through the security portal. "

--- Chunk 3 ---
"Password changes require identity verification.",

--- Chunk 4 ---
"Employees can work remotely up to three days per week, subject to manager approval.",

--- Chunk 5 ---
"Employees can contact the HR department for questions about employee benefits.",

--- Chunk 6 ---
"The company provides health insurance and retirement benefits to eligible employees."


In [ ]:
document_embeddings = model.encode(documents)

print(document_embeddings.shape)

(6, 384)


In [ ]:
def retrieve(query, documents, document_embeddings, top_k=2):
    query_embedding = model.encode(query)

    scores = cosine_similarity(
        [query_embedding],
        document_embeddings
    )[0]

    results = sorted(
        zip(documents, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return results[:top_k]

In [ ]:
context = retrieve(
    "How many annual leave days do employees get?",
    documents,
    document_embeddings,
    top_k=2
)

In [ ]:
for rank, (document, score) in enumerate(context, start=1):
    print(f"{rank}. {score:.4f} | {document}")

1. 0.7625 | "Employees are entitled to 20 days of annual leave each year.",
2. 0.4647 | "Employees can work remotely up to three days per week, subject to manager approval.",


LLM connection

In [ ]:
%pip install -qU google-genai

In [ ]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [ ]:
context = "\n\n".join(
    document for document, score in context
)

print(context)

"Employees are entitled to 20 days of annual leave each year.",

"Employees can work remotely up to three days per week, subject to manager approval.",


In [ ]:
def build_prompt(content,query):
  prompt = f"""
  You are a helpful company policy assistant.

  Answer the user's question using ONLY the context provided below.

  If the context does not contain enough information to answer the question,
  say that the information is not available in the provided context.

  Context:
  {context}

  Question:
  {query}
  """
  return prompt


In [ ]:
prompt = build_prompt(context, "How many annual leave days do employees get?")

In [ ]:
LLM_MODEL = "gemini-3.8-flash"
response = client.models.generate_content(
    model=LLM_MODEL,
    contents=prompt
)

print(response.text)

Employees are entitled to 20 days of annual leave each year.


In [ ]:
# Basic Chunking

# def chunk_text(text, chunk_size=30, overlap=5):
#     chunks = []

#     start = 0

#     while start < len(text):
#         end = start + chunk_size
#         chunk = text[start:end]

#         chunks.append(chunk)

#         start += chunk_size - overlap

#     return chunks

# document = """
# Employees are entitled to 20 days of annual leave each year.
# Employees must submit leave requests through the HR portal.
# Managers should approve leave requests within three business days.
# Employees can work remotely three days per week.
# """

# chunks = chunk_text(document)

# for i, chunk in enumerate(chunks):
#     print(f"Chunk {i + 1}:")
#     print(chunk)
#     print()